<a href="https://colab.research.google.com/github/RoselindSi/uapp/blob/main/notebooks/colab_lora_finetune_d3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRA fine-tune ESM2-650M on D3 — Google Colab

Run `scripts/12_lora_finetune_d3.py` on a free Colab T4 (≈30–45 min) instead of MPS Mac (≈2–3 hr).

## Before you start

1. **Set the runtime to a GPU**: `Runtime → Change runtime type → Hardware accelerator → T4 GPU` (Free tier is enough; A100 is ~3× faster on Colab Pro+).
2. **Upload your cache files to Google Drive once.**  In your Drive create a folder (default: `MyDrive/uapp_cache/`) and upload these two files from your Mac:
    - `cache/t2837_metadata.csv`  (≈ 1 MB, has `wtAA`, `mutAA`, `rel_rsa`, `pdb_code`, `split`, `mut_idx`, `sequence`, `ddG`, …)
    - `cache/t2837_bio_features_650m.pt`  (small, output of `scripts/06_build_bio_features.py`)

    The 650M cache file is *not* needed — LoRA re-runs the backbone every step.

3. Run the cells below top to bottom.  Final outputs are mirrored to `MyDrive/uapp_cache/outputs_lora_d3_650m/` so you can keep them between sessions.

## 1.  GPU sanity check + install deps

In [2]:
import torch, sys
print(f"Python:   {sys.version.split()[0]}")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    raise RuntimeError(
        "\n⚠️  No GPU detected.  In Colab go to Runtime → Change runtime type → T4 GPU, "
        "then re-run this cell."
    )

# Colab usually has transformers; peft may need installing.
!pip install -q -U peft transformers

Python:   3.12.13
PyTorch:  2.10.0+cu128
CUDA:     True
GPU:      NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM:     102.0 GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 153.2 MB/s eta 0:00:00


## 2.  Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Folder in your Drive containing t2837_metadata.csv and t2837_bio_features_650m.pt.
# Change this if you uploaded them somewhere else.
DRIVE_DIR = '/content/drive/MyDrive/uapp_cache'

import os
if not os.path.isdir(DRIVE_DIR):
    raise FileNotFoundError(
        f'{DRIVE_DIR} not found.  Create it in Drive and upload '
        '`t2837_metadata.csv` and `t2837_bio_features_650m.pt` first.'
    )
print('Drive contents:')
for f in sorted(os.listdir(DRIVE_DIR)):
    p = os.path.join(DRIVE_DIR, f)
    sz = os.path.getsize(p) / 1e6 if os.path.isfile(p) else 0
    print(f'  {f}  ({sz:.2f} MB)')

Mounted at /content/drive
Drive contents:
  Megascale_processed.csv  (649.78 MB)
  outputs_lora_d3_650m  (0.00 MB)
  t2837_bio_features_650m.pt  (0.08 MB)
  t2837_embeddings_v2_650m.pt  (26.89 MB)
  t2837_metadata.csv  (0.51 MB)


## 3.  Clone the repo + copy cache locally

Working out of `/content/uapp`.  We copy the small cache files from Drive to local SSD so the script's I/O is fast.

In [4]:
%cd /content
![ -d uapp ] || git clone https://github.com/RoselindSi/uapp.git
%cd uapp

# If you need a feature branch (script 12 not yet on main), uncomment:
# !git checkout claude/dreamy-curran-08f646

!git pull --ff-only

import shutil, os
os.makedirs('cache', exist_ok=True)
for fn in ['t2837_metadata.csv', 't2837_bio_features_650m.pt']:
    src = os.path.join(DRIVE_DIR, fn)
    dst = os.path.join('cache', fn)
    if not os.path.exists(src):
        raise FileNotFoundError(
            f'Missing {src} — please upload it to Drive first '
            f'(see the markdown at the top of this notebook).'
        )
    shutil.copy(src, dst)
    print(f'✓ {fn}  ({os.path.getsize(dst)/1e6:.2f} MB)')

/content
Cloning into 'uapp'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 237 (delta 106), reused 145 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (237/237), 8.68 MiB | 16.73 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/uapp
Already up to date.
✓ t2837_metadata.csv  (0.51 MB)
✓ t2837_bio_features_650m.pt  (0.08 MB)


## 4.  Run the LoRA fine-tune

Hyperparameters are sized for a free T4 (16 GB VRAM).  On A100 you can bump `--batch-size 32` and roughly halve walltime.

**Compute estimate**: ~1–2 min/epoch on T4, ~25–40 min total for 20 epochs.

In [ ]:
!python scripts/12_lora_finetune_d3.py \
    --metadata-csv cache/t2837_metadata.csv \
    --bio-feats    cache/t2837_bio_features_650m.pt \
    --out          outputs/lora_d3_650m \
    --device       cuda \
    --batch-size   16 \
    --max-epochs   20 \
    --patience     5 \
    --lr           5e-4

## 5.  Inspect the results

In [ ]:
import json
from pathlib import Path

OUT = Path('outputs/lora_d3_650m')
metrics = json.loads((OUT / 'test_metrics.json').read_text())
log     = json.loads((OUT / 'training_log.json').read_text())

print('=' * 78)
print(f"LoRA fine-tune of ESM2-650M on D3 (RSA + chemistry)")
print('=' * 78)
print(f"Trained {len(log)} epochs.  Best val_loss = "
      f"{min(e['val_loss'] for e in log):.4f}")

print(f"\nTest metrics (n = {metrics['n']}):")
for k, v in metrics.items():
    if k == 'n': continue
    if isinstance(v, float):
        print(f"  {k:<12} = {v:+.4f}")
    else:
        print(f"  {k:<12} = {v}")

print("\nCompare against the frozen-backbone D3 baseline (script 09 fixed-split):")
print("  Frozen 650M D3:  RMSE 1.50   NLL 1.89   ICE 0.069   Spearman 0.333")
print('=' * 78)

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

epochs = [e['epoch'] for e in log]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, [e['train_loss'] for e in log], 'o-', label='train', color='#0D7377')
ax.plot(epochs, [e['val_loss']   for e in log], 's-', label='val',   color='#E8913A')
ax.set_xlabel('epoch'); ax.set_ylabel('Student-t NLL')
ax.set_title('LoRA D3-650M training curves')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 6.  Save outputs back to Drive

This mirrors `outputs/lora_d3_650m/` to your Drive so you can pull the LoRA adapter, head weights, and metrics down to your Mac.  Total ~50 MB.

In [ ]:
import shutil, os
DST = os.path.join(DRIVE_DIR, 'outputs_lora_d3_650m')
shutil.copytree('outputs/lora_d3_650m', DST, dirs_exist_ok=True)
for root, _, files in os.walk(DST):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {os.path.relpath(p, DRIVE_DIR):<55}  {os.path.getsize(p)/1e6:6.2f} MB')
print(f"\n✓ Mirrored to: {DST}")

## 7.  (Optional) Re-build bio features on Colab from scratch

If you only uploaded `t2837_metadata.csv` and *not* the bio-features file, run this cell to build it on Colab using `scripts/06_build_bio_features.py`.  This takes a few seconds.

You also need the embedding cache file for this to work — script 06 validates row counts against it.  If the embedding cache lives in Drive too:

```python
shutil.copy(os.path.join(DRIVE_DIR, 't2837_embeddings_v2_650m.pt'), 'cache/')
```

Otherwise, build embeddings on Colab too with `scripts/01_cache_embeddings_esm_v2.py --esm-model facebook/esm2_t33_650M_UR50D --device cuda` (~10 min on T4).

In [ ]:
# Only run this if you didn't upload t2837_bio_features_650m.pt to Drive
# !python scripts/06_build_bio_features.py \
#     --metadata-csv cache/t2837_metadata.csv \
#     --embeddings   cache/t2837_embeddings_v2_650m.pt \
#     --rsa-col      rel_rsa \
#     --out          cache/t2837_bio_features_650m.pt

## 8.  (Optional) Re-run the analysis pipeline on Colab

Now that you're on a GPU, you can also redo the multi-seed and K-fold CV runs much faster:

```bash
# Multi-seed (script 09) — ~3 min on T4
!python scripts/09_multiseed_experiment_d.py \
    --embeddings cache/t2837_embeddings_v2_650m.pt \
    --bio-feats  cache/t2837_bio_features_650m.pt \
    --out        outputs/multiseed_650m \
    --device cuda --seeds 0 1 2 3 4 5 6 7

# K-fold CV (script 11) — ~5 min on T4 with K=5 folds × 5 seeds = 25 paired obs
!python scripts/11_kfold_cv_track_d.py \
    --embeddings cache/t2837_embeddings_v2_650m.pt \
    --bio-feats  cache/t2837_bio_features_650m.pt \
    --out        outputs/cv_650m_more_seeds \
    --device cuda --folds 5 --seeds 0 1 2 3 4
```

On a GPU, the bottleneck shifts from compute to test-set noise.  More seeds **directly** tighten your statistical confidence intervals.

In [6]:
%cd /content/uapp
!git fetch origin
!git checkout claude/dreamy-curran-08f646
!git pull --ff-only

# 验证 script 15 现在存在
!ls scripts/15_megascale_to_t2837_format.py scripts/16_pretrain_and_finetune.py

/content/uapp
Already on 'claude/dreamy-curran-08f646'
Your branch is up to date with 'origin/claude/dreamy-curran-08f646'.
Already up to date.
scripts/15_megascale_to_t2837_format.py  scripts/16_pretrain_and_finetune.py


In [10]:
# ──────────────────────────────────────────────────────────────────────
# Cell C5a (替换之前的版本)  —  Direct download from Zenodo on Colab
# ──────────────────────────────────────────────────────────────────────
import urllib.request, zipfile, pandas as pd
from pathlib import Path

ZENODO_URL  = 'https://zenodo.org/records/7992926/files/Processed_K50_dG_datasets.zip'
ZIP_PATH    = Path('/content/uapp/data/Processed_K50_dG_datasets.zip')   # Colab local disk
EXTRACT_DIR = Path('/content/uapp/data/megascale_extracted')
OUT_CSV     = Path('/content/uapp/data/Megascale_processed.csv')

# 1) 直接 wget 到 Colab 本地 disk (~30 sec)
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size < 1e9:
    print(f"Downloading {ZENODO_URL} ...")
    !wget -q --show-progress -O {ZIP_PATH} {ZENODO_URL}
    print(f"\n✓ Downloaded ({ZIP_PATH.stat().st_size / 1e6:.0f} MB)")
else:
    print(f"Already downloaded: {ZIP_PATH}")

# 2) Extract
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(EXTRACT_DIR)
print(f"Extracted to {EXTRACT_DIR}")

# 3) Concatenate per-protein CSVs
csv_files = sorted(EXTRACT_DIR.glob('**/*.csv'))
print(f"Found {len(csv_files)} per-protein CSV files")

dfs, skipped = [], 0
for csv in csv_files:
    try:
        df = pd.read_csv(csv)
        df['name'] = csv.stem
        dfs.append(df)
    except Exception:
        skipped += 1; continue

big = pd.concat(dfs, ignore_index=True)
print(f"Combined: {len(big):,} rows × {big.shape[1]} cols  (skipped {skipped})")
print(f"Columns: {list(big.columns)[:12]}")
if 'mut_type' in big.columns:
    print(f"mut_type sample: {big['mut_type'].value_counts().head(5).to_dict()}")
if 'dG_ML' in big.columns:
    print(f"dG_ML range: [{big['dG_ML'].min():.2f}, {big['dG_ML'].max():.2f}]")

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
big.to_csv(OUT_CSV, index=False)
print(f"\n✓ Saved {OUT_CSV}  ({OUT_CSV.stat().st_size/1e6:.1f} MB)")

# 4) (可选) 把 *处理后的* 小 CSV 备份到 Drive，下次 session 不用重下大 ZIP
import shutil, os
DRIVE_BACKUP = Path('/content/drive/MyDrive/uapp_cache/Megascale_processed.csv')
if Path('/content/drive/MyDrive').exists():
    DRIVE_BACKUP.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(OUT_CSV, DRIVE_BACKUP)
    print(f"✓ Backed up processed CSV to Drive: {DRIVE_BACKUP} "
          f"({DRIVE_BACKUP.stat().st_size/1e6:.1f} MB)")

/content/uapp/data/ 100%[===================>] 966.72M  2.99MB/s    in 11m 42s 

✓ Downloaded (1014 MB)
Extracted to /content/uapp/data/megascale_extracted
Found 10 per-protein CSV files


/tmp/ipykernel_3695/2403063549.py:34: DtypeWarning: Columns (30,31,36) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv)


Combined: 2,619,327 rows × 73 cols  (skipped 5)
Columns: ['K50_corr', 'dg_corr', 'recon_corr', 'NA_num', 'max_ep_dG', 'wt_ep_dG', 'double_mut_name', 'name', 'aa_seq', 'frac_NA', 'raw_corr', 'slope']
mut_type sample: {'wt': 2348, 'insG21': 469, 'insA22': 469, 'insA13': 469, 'insA27': 469}


TypeError: '<=' not supported between instances of 'float' and 'str'

In [15]:
# # Quick fix — `big` is already in memory; just clean and save it
# import pandas as pd, numpy as np
# from pathlib import Path

# OUT_CSV = Path('/content/uapp/data/Megascale_processed.csv')

# # Coerce numeric columns so script 15 doesn't trip on string 'NA' values
# for c in ('dG_ML', 'ddG_ML', 'dG', 'ddG', 'dg_corr', 'wt_ep_dG'):
#     if c in big.columns:
#         big[c] = pd.to_numeric(big[c], errors='coerce')

# # Drop rows where dG_ML is NaN (those are unmeasurable variants — useless for us)
# n_before = len(big)
# if 'dG_ML' in big.columns:
#     big = big.dropna(subset=['dG_ML']).reset_index(drop=True)
# print(f"Dropped {n_before - len(big):,} rows with missing dG_ML")
# print(f"Final: {len(big):,} rows")

# # Save
# OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
# big.to_csv(OUT_CSV, index=False)
# print(f"\n✓ Saved {OUT_CSV}  ({OUT_CSV.stat().st_size/1e6:.1f} MB)")

# # Diagnostics
# print(f"\nColumns ({big.shape[1]} total):")
# print(f"  {list(big.columns)}")

# if 'mut_type' in big.columns:
#     sav_mask = big['mut_type'].astype(str).str.match(r'^[A-Z]\d+[A-Z]$')
#     n_sav = int(sav_mask.sum())
#     n_wt  = int((big['mut_type'].astype(str).str.lower() == 'wt').sum())
#     n_ins = int(big['mut_type'].astype(str).str.startswith('ins').sum())
#     n_del = int(big['mut_type'].astype(str).str.startswith('del').sum())
#     print(f"\nmut_type breakdown:")
#     print(f"  SAVs (e.g. 'I7L'):  {n_sav:,}    ← script 15 keeps these")
#     print(f"  wild-type rows:     {n_wt:,}    ← used to compute ΔΔG = ΔG_mut - ΔG_wt")
#     print(f"  insertions:         {n_ins:,}    ← script 15 drops")
#     print(f"  deletions:          {n_del:,}    ← script 15 drops")

# if 'dG_ML' in big.columns:
#     print(f"\ndG_ML range: [{big['dG_ML'].min():.2f}, {big['dG_ML'].max():.2f}]   "
#           f"mean={big['dG_ML'].mean():.2f}")

# # Backup the *small* processed CSV to Drive (much faster than uploading the 1 GB ZIP)
# import shutil
# DRIVE_BACKUP = Path('/content/drive/MyDrive/uapp_cache/Megascale_processed.csv')
# if Path('/content/drive/MyDrive').exists():
#     DRIVE_BACKUP.parent.mkdir(parents=True, exist_ok=True)
#     shutil.copy(OUT_CSV, DRIVE_BACKUP)
#     print(f"\n✓ Backed up to Drive: {DRIVE_BACKUP}  "
#           f"({DRIVE_BACKUP.stat().st_size/1e6:.1f} MB)")

In [16]:
# # 转成 T2837 格式 (~1 min, drops indels and multi-mutants)
# !python scripts/15_megascale_to_t2837_format.py \
#     --megascale-csv /content/uapp/data/Megascale_processed.csv \
#     --out cache/megascale_metadata.csv \
#     --val-frac 0.10 --seed 42

In [17]:
# # 1. Cache ESM2-650M embeddings (uses our existing v2 caching script):
# !python scripts/01_cache_embeddings_esm_v2.py \
#       --t2837-csv cache/megascale_metadata.csv \
#       --out cache/megascale_embeddings_650m.pt \
#       --esm-model facebook/esm2_t33_650M_UR50D \
#       --device cuda --seed 42

#   # 2. Build extended bio features (k=13: chemistry + sequence-struct):
# !python scripts/06_build_bio_features.py \
#       --metadata-csv cache/megascale_metadata.csv \
#       --embeddings   cache/megascale_embeddings_650m.pt \
#       --out          cache/megascale_bio_features_650m_extended.pt \
#       --include-extended

#   # 3. Run the pretrain+finetune script (script 16):
# !python scripts/16_pretrain_and_finetune.py \
#       --megascale-emb cache/megascale_embeddings_650m.pt \
#       --megascale-bio cache/megascale_bio_features_650m_extended.pt \
#       --t2837-emb     cache/t2837_embeddings_v2_650m.pt \
#       --t2837-bio     cache/t2837_bio_features_650m_extended.pt \
#       --out           outputs/megascale_pretrain

In [18]:
# ───────────────────────────────────────────────────────────────────────────
# Step 1 — Re-build Megascale_processed.csv WITHOUT overwriting `name`
# ───────────────────────────────────────────────────────────────────────────
import zipfile, pandas as pd
from pathlib import Path

EXTRACT_DIR = Path('/content/uapp/data/megascale_extracted')
OUT_CSV     = Path('/content/uapp/data/Megascale_processed.csv')

csv_files = sorted(EXTRACT_DIR.glob('**/*.csv'))
print(f"Found {len(csv_files)} CSV files")

dfs, skipped = [], 0
for csv in csv_files:
    try:
        df = pd.read_csv(csv, low_memory=False)
        if 'name' not in df.columns:        # only fall back to filename if missing
            df['name'] = csv.stem
        df['_source_file'] = csv.stem       # for debugging only
        dfs.append(df)
    except Exception:
        skipped += 1; continue

big = pd.concat(dfs, ignore_index=True)
for c in ('dG_ML', 'ddG_ML', 'dG', 'ddG', 'dg_corr', 'wt_ep_dG'):
    if c in big.columns:
        big[c] = pd.to_numeric(big[c], errors='coerce')
n_before = len(big)
big = big.dropna(subset=['dG_ML']).reset_index(drop=True)
print(f"Final: {len(big):,} rows  (dropped {n_before-len(big):,} NaN dG_ML)")
print(f"Unique 'name':    {big['name'].nunique()}")
print(f"Unique 'WT_name': {big['WT_name'].nunique()}")

big.to_csv(OUT_CSV, index=False)
print(f"\n✓ Saved {OUT_CSV} ({OUT_CSV.stat().st_size/1e6:.0f} MB)")

# ───────────────────────────────────────────────────────────────────────────
# Step 2 — Pull the script 15 fix
# ───────────────────────────────────────────────────────────────────────────
%cd /content/uapp
!git checkout claude/dreamy-curran-08f646
!git pull --ff-only

# ───────────────────────────────────────────────────────────────────────────
# Step 3 — Re-run script 15 with the WT-seq lookup
# ───────────────────────────────────────────────────────────────────────────
!python scripts/15_megascale_to_t2837_format.py \
    --megascale-csv data/Megascale_processed.csv \
    --out cache/megascale_metadata.csv \
    --val-frac 0.10 --seed 42

Found 10 CSV files
Final: 675,467 rows  (dropped 1,943,860 NaN dG_ML)
Unique 'name':    675277
Unique 'WT_name': 479

✓ Saved /content/uapp/data/Megascale_processed.csv (664 MB)
/content/uapp
Already on 'claude/dreamy-curran-08f646'
Your branch is up to date with 'origin/claude/dreamy-curran-08f646'.
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.30 KiB | 2.30 MiB/s, done.
From https://github.com/RoselindSi/uapp
   3d99218..b1a30fc  claude/dreamy-curran-08f646 -> origin/claude/dreamy-curran-08f646
Updating 3d99218..b1a30fc
Fast-forward
 scripts/15_megascale_to_t2837_format.py | 92 ++++++++++++++++++++-------------
 1 file changed, 56 insertions(+), 36 deletions(-)
13:32:31 [INFO] uapp: Reading Megascale CSV: data/Megascale_processed.csv
/content/uapp/scripts/15_megascale_to_t2837_format.py:144: DtypeWarn

In [19]:
# Quick smoke test —只用 4K samples (1% of full)
import pandas as pd
md = pd.read_csv('cache/megascale_metadata.csv')
md_small = md.sample(n=4000, random_state=42)
md_small.to_csv('cache/megascale_metadata_smoke.csv', index=False)
print(f"Smoke test: {len(md_small)} rows, {md_small['pdb_code'].nunique()} proteins")

!python scripts/01_cache_embeddings_esm_v2.py \
    --t2837-csv cache/megascale_metadata_smoke.csv \
    --out cache/megascale_embeddings_650m_smoke.pt \
    --esm-model facebook/esm2_t33_650M_UR50D \
    --device cuda --seed 42
# 这一步如果 OK 大概 1-2 min, 然后再上全量

Smoke test: 4000 rows, 412 proteins
13:36:25 [INFO] uapp: device: cuda
13:36:25 [INFO] uapp: loading T2837 from cache/megascale_metadata_smoke.csv
13:36:25 [INFO] uapp: loaded 4000 mutations
13:36:25 [INFO] uapp: resolving mutation positions to sequence indices...
13:36:25 [INFO] uapp: position resolution: 4000/4000 mapped (dropped 0 unmappable)
13:36:25 [INFO] uapp:   train: 2773 mutations from 290 proteins
13:36:25 [INFO] uapp:   val: 625 mutations from 61 proteins
13:36:25 [INFO] uapp:   test: 602 mutations from 61 proteins
13:36:28 [INFO] uapp: loading ESM-2: facebook/esm2_t33_650M_UR50D
13:36:28 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/esm2_t33_650M_UR50D/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
13:36:28 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
13:36:28 [INFO] httpx: HTTP Request: HEAD https://hu

In [20]:
# Step 4 — Cache ESM2-650M embeddings for Megascale (~5-10 min on T4)
%cd /content/uapp
!python scripts/01_cache_embeddings_esm_v2.py \
    --t2837-csv cache/megascale_metadata.csv \
    --out cache/megascale_embeddings_650m.pt \
    --esm-model facebook/esm2_t33_650M_UR50D \
    --device cuda --seed 42

/content/uapp
13:36:57 [INFO] uapp: device: cuda
13:36:57 [INFO] uapp: loading T2837 from cache/megascale_metadata.csv
13:36:57 [INFO] uapp: loaded 389068 mutations
13:36:57 [INFO] uapp: resolving mutation positions to sequence indices...
13:37:02 [INFO] uapp: position resolution: 389068/389068 mapped (dropped 0 unmappable)
13:37:02 [INFO] uapp:   train: 276569 mutations from 290 proteins
13:37:02 [INFO] uapp:   val: 55183 mutations from 61 proteins
13:37:02 [INFO] uapp:   test: 57316 mutations from 61 proteins
13:37:04 [INFO] uapp: loading ESM-2: facebook/esm2_t33_650M_UR50D
13:37:04 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/facebook/esm2_t33_650M_UR50D/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
13:37:04 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
13:37:04 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api

In [26]:
# 1. 把 script 01 写出来的 (Megascale-content) metadata 重命名，避免被未来误用
# import shutil
# shutil.move('cache/t2837_metadata.csv', 'cache/megascale_processed_metadata.csv')
# print("✓ Renamed: cache/megascale_processed_metadata.csv")
# print("  (this is the Megascale metadata with splits matching the embedding cache)")

# 2. Build Megascale bio features using the renamed metadata
!python scripts/06_build_bio_features.py \
    --metadata-csv cache/megascale_processed_metadata.csv \
    --embeddings   cache/megascale_embeddings_650m.pt \
    --out          cache/megascale_bio_features_650m_extended.pt \
    --include-extended

# 3. Pretrain + finetune
!python scripts/16_pretrain_and_finetune.py \
    --megascale-emb cache/megascale_embeddings_650m.pt \
    --megascale-bio cache/megascale_bio_features_650m_extended.pt \
    --t2837-emb     cache/t2837_embeddings_v2_650m.pt \
    --t2837-bio     cache/t2837_bio_features_650m_extended.pt \
    --out           outputs/megascale_pretrain \
    --device cuda

13:51:28 [INFO] uapp: Embedding-cache split sizes: {'train': 276569, 'val': 55183, 'test': 57316}
13:51:29 [INFO] uapp: Loaded 389068 metadata rows from cache/megascale_processed_metadata.csv
13:51:29 [INFO] uapp: CSV split sizes: {'test': 57316, 'train': 276569, 'val': 55183}
13:51:31 [INFO] uapp:   train: bio-feature tensor shape (276569, 13)
13:51:31 [INFO] uapp:   val: bio-feature tensor shape (55183, 13)
13:51:31 [INFO] uapp:   test: bio-feature tensor shape (57316, 13)
13:51:31 [INFO] uapp: Saved aligned bio-feature tensor to cache/megascale_bio_features_650m_extended.pt
13:51:31 [INFO] uapp: Feature names (k=13): ['rsa', 'blosum62', 'grantham', 'delta_charge', 'delta_polarity', 'delta_hydrophobicity', 'delta_volume', 'delta_helix_propensity', 'delta_sheet_propensity', 'local_entropy', 'local_hydrophobic_count', 'local_charged_count', 'position_relative']
13:51:33 [INFO] uapp: Device: cuda
13:51:33 [INFO] uapp: 
══════════════ PRETRAIN on Megascale ══════════════
13:51:34 [INFO] 

In [24]:
# import os, shutil
# candidates = [
#     '/content/drive/MyDrive/uapp_cache/t2837_embeddings_v2_650m.pt',
#     '/content/drive/MyDrive/uapp_cache/t2837_embeddings_v2.pt',  # 8M version, fallback
# ]
# for src in candidates:
#     if os.path.exists(src):
#         dst = f'cache/{os.path.basename(src)}'
#         shutil.copy(src, dst)
#         print(f'✓ Restored: {dst}  ({os.path.getsize(dst)/1e6:.1f} MB)')
#         break
# else:
#     print('Not in Drive — use Option B or C')

✓ Restored: cache/t2837_embeddings_v2_650m.pt  (26.9 MB)


In [28]:
# Try restoring from Drive
import os, shutil
candidates = [
    'cache/t2837_bio_features_650m_extended.pt',
    'cache/t2837_bio_features_650m_dssp.pt',  # also useful — for D6 ensemble later
    'cache/t2837_metadata.csv',                # in case you backed up T2837 metadata before
]
for fn in candidates:
    src = f'/content/drive/MyDrive/uapp_cache/{os.path.basename(fn)}'
    if os.path.exists(src) and not os.path.exists(fn):
        shutil.copy(src, fn)
        print(f'✓ Restored {fn}  ({os.path.getsize(fn)/1e6:.1f} MB)')
    elif os.path.exists(fn):
        print(f'  already exists: {fn}')
    else:
        print(f'  not in Drive: {os.path.basename(src)}')

  not in Drive: t2837_bio_features_650m_extended.pt
  not in Drive: t2837_bio_features_650m_dssp.pt
✓ Restored cache/t2837_metadata.csv  (0.5 MB)


In [29]:
# Option B — rebuild T2837 metadata + bio features in one go
# 这一步会重新生成 cache/t2837_embeddings_v2_650m.pt 但是数值跟你刚恢复的版本一致
# (same seed, same model). 如果你想避开这步重 cache, 见下面 Option C.

# 先确保 StabilityOracle T2837 csv 在
!ls StabilityOracle/data/datasets/T2837.csv 2>/dev/null \
    || git clone --depth 1 https://github.com/CnDevs/StabilityOracle.git

# 重新跑 script 01 (~5 min on T4) — 会同时生成 t2837_metadata.csv
!python scripts/01_cache_embeddings_esm_v2.py \
    --t2837-csv StabilityOracle/data/datasets/T2837.csv \
    --out cache/t2837_embeddings_v2_650m.pt \
    --metadata-out cache/t2837_metadata.csv \
    --esm-model facebook/esm2_t33_650M_UR50D \
    --device cuda --seed 42

# 再跑 script 06 (~30 秒) 生成 bio features
!python scripts/06_build_bio_features.py \
    --metadata-csv cache/t2837_metadata.csv \
    --embeddings   cache/t2837_embeddings_v2_650m.pt \
    --out          cache/t2837_bio_features_650m_extended.pt \
    --include-extended

# 再跑 script 16 fine-tune
!python scripts/16_pretrain_and_finetune.py \
    --megascale-emb cache/megascale_embeddings_650m.pt \
    --megascale-bio cache/megascale_bio_features_650m_extended.pt \
    --t2837-emb     cache/t2837_embeddings_v2_650m.pt \
    --t2837-bio     cache/t2837_bio_features_650m_extended.pt \
    --out           outputs/megascale_pretrain \
    --device cuda

Cloning into 'StabilityOracle'...
fatal: could not read Username for 'https://github.com': No such device or address
usage: 01_cache_embeddings_esm_v2.py [-h] --t2837-csv T2837_CSV --out OUT
                                     [--window-size WINDOW_SIZE]
                                     [--val-fraction VAL_FRACTION]
                                     [--test-fraction TEST_FRACTION]
                                     [--seed SEED] [--device DEVICE]
                                     [--esm-model ESM_MODEL]
01_cache_embeddings_esm_v2.py: error: unrecognized arguments: --metadata-out cache/t2837_metadata.csv
14:11:18 [INFO] uapp: Embedding-cache split sizes: {'train': 1395, 'val': 1019, 'test': 170}
14:11:18 [INFO] uapp: Loaded 2584 metadata rows from cache/t2837_metadata.csv
14:11:18 [INFO] uapp: CSV split sizes: {'test': 170, 'train': 1395, 'val': 1019}
14:11:18 [INFO] uapp:   train: bio-feature tensor shape (1395, 13)
14:11:18 [INFO] uapp:   val: bio-feature tensor shape (10

In [21]:
# Step 5 — Bio features (fast, < 1 min)
!python scripts/06_build_bio_features.py \
    --metadata-csv cache/megascale_metadata.csv \
    --embeddings   cache/megascale_embeddings_650m.pt \
    --out          cache/megascale_bio_features_650m_extended.pt \
    --include-extended

# Step 6 — Pretrain on Megascale + fine-tune on T2837 (~20-30 min)
!python scripts/16_pretrain_and_finetune.py \
    --megascale-emb cache/megascale_embeddings_650m.pt \
    --megascale-bio cache/megascale_bio_features_650m_extended.pt \
    --t2837-emb     cache/t2837_embeddings_v2_650m.pt \
    --t2837-bio     cache/t2837_bio_features_650m_extended.pt \
    --out           outputs/megascale_pretrain \
    --device        cuda

13:38:34 [INFO] uapp: Embedding-cache split sizes: {'train': 276569, 'val': 55183, 'test': 57316}
13:38:34 [INFO] uapp: Loaded 389068 metadata rows from cache/megascale_metadata.csv
13:38:34 [INFO] uapp: CSV split sizes: {'train': 351180, 'val': 37888}
usage: 06_build_bio_features.py [-h] [--metadata-csv METADATA_CSV]
                                --embeddings EMBEDDINGS --out OUT
                                [--rsa-col RSA_COL] [--include-indicators]
                                [--include-extended] [--log-level LOG_LEVEL]
06_build_bio_features.py: error: Row-count mismatch for split 'train': embedding cache has 276569 rows but metadata CSV has 351180.  Make sure you are passing --metadata-csv to the *processed* CSV saved by 01_cache_embeddings_esm_v2.py, not the raw T2837.csv.
13:38:36 [INFO] uapp: Device: cuda
13:38:36 [INFO] uapp: 
══════════════ PRETRAIN on Megascale ══════════════
Traceback (most recent call last):
  File "/content/uapp/scripts/16_pretrain_and_finetune.py

In [5]:
# %cd /content/uapp
# !git checkout claude/dreamy-curran-08f646
# !git pull --ff-only

# !python scripts/15_megascale_to_t2837_format.py \
#     --megascale-csv data/Megascale_processed.csv \
#     --out cache/megascale_metadata.csv \
#     --val-frac 0.10 --seed 42

/content/uapp
Branch 'claude/dreamy-curran-08f646' set up to track remote branch 'claude/dreamy-curran-08f646' from 'origin'.
Switched to a new branch 'claude/dreamy-curran-08f646'
Already up to date.
13:00:37 [INFO] uapp: Reading Megascale CSV: data/Megascale_processed.csv
Traceback (most recent call last):
  File "/content/uapp/scripts/15_megascale_to_t2837_format.py", line 315, in <module>
    main()
  File "/content/uapp/scripts/15_megascale_to_t2837_format.py", line 144, in main
    df = pd.read_csv(args.megascale_csv)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.